In [1]:
! pwd

/lessons


## Урок 4



### Задание 1

In [2]:

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .master("local") \
                    .appName("Learning DataFrames") \
                    .getOrCreate()

data = [('2021-01-04', 3744, 63, 322),
        ('2021-01-04', 2434, 21, 382),
        ('2021-01-04', 2434, 32, 159),
        ('2021-01-04', 3744, 32, 159),
        ('2021-01-04', 4342, 32, 159),
        ('2021-01-04', 4342, 12, 259),
        ('2021-01-04', 5677, 12, 259),
        ('2021-01-04', 5677, 23, 499)
]

columns = ['dt', 'user_id', 'product_id', 'purchase_amount']
df = spark.createDataFrame(data=data, schema=columns)

/opt/spark/conf/spark-env.sh: line 31: hadoop: command not found


26/08/18 20:05:45 WARN Utils: Your hostname, fv4im62q1f2ij1439k95 resolves to a loopback address: 127.0.1.1; using 10.130.0.37 instead (on interface eth0)
26/08/18 20:05:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
df.printSchema()

root
 |-- dt: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- purchase_amount: long (nullable = true)



In [7]:
data = [
    ("Max", 55),
    ("Yan", 53),
    ("Dmitry", 54),
    ("Ann", 25)
]

columns = ['Name', 'Age']
df = spark.createDataFrame(data=data, schema=columns)
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)



### Задание 2

In [9]:
df = spark.read.load(
    path="/user/master/data/events/date=2022-05-25",
    format='json'
)

In [10]:
df.show(10)

+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
+--------------------+------------+
only showing top 10 rows



### Задание 3

In [11]:
! hdfs dfs -ls /user/master/data/snapshots

/usr/bin/sh: line 1: hdfs: command not found


In [12]:
! whereis hdfs

hdfs: /opt/hadoop/bin/hdfs


In [17]:
! /opt/hadoop/bin/hdfs dfs -ls /user/master/data/snapshots/channels/actual/

Found 2 items
-rw-r--r--   3 ubuntu hadoop          0 2025-11-06 03:54 /user/master/data/snapshots/channels/actual/_SUCCESS
-rw-r--r--   3 ubuntu hadoop    4295876 2025-11-06 03:54 /user/master/data/snapshots/channels/actual/part-00000-beae86b3-af54-4415-8f29-31dd79cfe178-c000.snappy.parquet


In [18]:
df = spark.read.load(
    "/user/master/data/snapshots/channels/actual/", 
    format="parquet"
)




In [21]:
df.write.option("header", True) \
  .partitionBy("channel_type") \
  .mode("append") \
  .parquet("/user/s18314377/analytics/test")

In [22]:
events = spark.read.parquet("/user/s18314377/analytics/test") \
              .select("channel_type") \
              .orderBy("channel_type") \
              .distinct() \
              .show()

+------------+
|channel_type|
+------------+
|       river|
|     channel|
+------------+

